# AEGIS — Day 2 | Forecasting, RAG and managed streaming

**Prerequisite:** Complete Day 1; have an active Colab runtime with extracted repository, authenticated project, and BigQuery reference tables.

**Cost gates:** BigQuery ML and Vertex AI can incur charges; Dataflow runs workers until stopped. Review free-trial credit balance before each advanced section.

## 1. Repeat Day 1 setup (runtime resets)

If Colab disconnected, rerun Day 1 cells to upload ZIP, install packages, authenticate and set `PROJECT_ID`. Do not republish events unless you want additional data.

In [ ]:
# Work in /content/AEGIS_GCP_RealTime_Portfolio and set PROJECT_ID in Day 1.
import os
print('Current folder:', os.getcwd())

## 2. Calculate supplier delay scenarios (no cloud charges)

In [ ]:
import pandas as pd
from aegis.risk import decision_replay
scenarios = pd.DataFrame(decision_replay())
display(scenarios)

## 3. BigQuery ML (optional, may use credits)

Open `sql/03_bigquery_ml_optional.sql` in [BigQuery Studio](https://console.cloud.google.com/bigquery). Replace `{{PROJECT_ID}}` with your real ID. Run the `CREATE MODEL` statement first, then the evaluation statement, then baseline MAE. Do not report accuracy until you've actually run it.

## 4. Grounded supplier policy retrieval (free baseline)

This baseline uses TF-IDF vectors. It is not a neural embedding model; it shows the retrieval-and-grounding workflow.

In [ ]:
from aegis.retrieval import search_policies, draft_evidence_prompt
question = 'What escalation is required for a supplier shipment delayed seven days?'
found = search_policies(question)
display(pd.DataFrame([{'document':d['source'],'similarity':d['score']} for d in found]))
prompt = draft_evidence_prompt(question,found,decision_replay())
print(prompt[:1600])

## 5. Neural embeddings (optional, free Colab CPU; model download required)

In [ ]:
# Uncomment if you want to download an embedding model and perform semantic vector retrieval:
# %pip install -q -r requirements-rag-optional.txt
# from aegis.semantic_retrieval import semantic_search
# semantic_docs = semantic_search(question)
# print([(d['source'],round(d['score'],3)) for d in semantic_docs])

## 6. Gemini via Vertex AI (optional paid service)

Only after checking that Vertex AI billing, region and model access are available:

```python
from aegis.retrieval import ask_vertex_gemini
answer = ask_vertex_gemini(prompt, PROJECT_ID)
print(answer)
```

This API call is deliberately NOT run by default. Model pricing and access can change.

## 7. Managed Dataflow (optional, credit-consuming)

Read `docs/DAY2_ADVANCED.md` carefully. For Dataflow, create a *separate pull subscription* and a GCS staging bucket, configure IAM, and run the Beam pipeline only while actively testing. Drain the Dataflow job in the Google Console after capturing proof.

Do not claim a managed Dataflow deployment until the dashboard shows an actual running/terminated job and your output tables contain events.